In [1]:
from utility.plot_spectrum import plot_spectrum
from utility.adjust_redshift import adjust_redshift

from sparcl.client import SparclClient
from dl import queryClient as qc
from astropy.coordinates import SkyCoord, Angle
from astropy import units as u
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json
import os


plt.rcParams.update({"legend.frameon":False})

hamann_erq_path = "data/core-ERQ-stats-coords.csv"
search_log_path = "data/local/hamann-objects-desi-search-log.json"

### prepare Hamann ERQ data

#### clean core ERQ dataframe

In [ ]:
# hamann_erq = pd.read_csv("data/Hamann2017-core-ERQ-stats.csv", skiprows=[1])


# ## rename columns
# col_rename_dic = {
#             "Quasar name":"SDSS_id",
#             "N V/C IV":"NV/CIV",
#             "αλ":"UV_slope",
#             "E(B - V)":"E(B-V)",
#             "i–W3":"i-W3"
#             }

# hamann_erq = hamann_erq.rename(columns=col_rename_dic)


# ## change weird - into nan
# hamann_erq = hamann_erq.replace("–",np.nan)
# hamann_erq = hamann_erq.replace("-",np.nan)


# ## remove the letters from the identifier
# hamann_erq["SDSS_id"] = hamann_erq["SDSS_id"].apply(lambda x: x[:-1] if x[-1].isalpha() else x)


# ## split up uncertainties to new columns, turn into floats
# hamann_erq["REW_unc"] = hamann_erq["REW"].apply(lambda x: x.split("±")[1]).astype("float")
# hamann_erq["REW"] = hamann_erq["REW"].apply(lambda x: x.split("±")[0]).astype("float")

# hamann_erq["FWHM_unc"] = hamann_erq["FWHM"].apply(lambda x: x.split("±")[1]).astype("float")
# hamann_erq["FWHM"] = hamann_erq["FWHM"].apply(lambda x: x.split("±")[0]).astype("float")

# hamann_erq["E(B-V)_unc"] = hamann_erq["E(B-V)"].astype(str).apply(lambda x: x.split("±"))  # has nan values hence the extra step inbetween
# hamann_erq["E(B-V)"] = hamann_erq["E(B-V)_unc"].apply(lambda x: x[0]).astype("float")
# hamann_erq["E(B-V)_unc"] = hamann_erq["E(B-V)_unc"].apply(lambda x: x[1] if len(x) > 1 else x[0]).astype("float")

# hamann_erq["FIRST"] = hamann_erq["FIRST"].astype("float")


# ## change column order
# hamann_erq = hamann_erq[['SDSS_id', 'ze', 'i', 'i-W3', 'REW', 'REW_unc', 'FWHM', 'FWHM_unc', 'kt80', 'NV/CIV', 'BAL', 'UV_slope', 'E(B-V)', 'E(B-V)_unc', 'FIRST']]

# hamann_erq

,SDSS_id,ze,i,i-W3,REW,REW_unc,FWHM,FWHM_unc,kt80,NV/CIV,BAL,UV_slope,E(B-V),E(B-V)_unc,FIRST
0,J000610.67+121501.2,2.31,22.1,8.0,107.0,6.0,4540.0,200.0,0.37,1.88,0,-0.20,0.66,0.01,0.0
1,J000746.19+122223.9,2.43,21.3,4.9,220.0,6.0,2433.0,71.0,0.28,0.39,0,-0.17,0.19,0.01,0.0
2,J002400.25+245031.9,2.81,21.6,6.0,144.0,10.0,3523.0,195.0,0.37,2.46,0,-1.14,0.20,0.02,NaN
3,J004713.21+264024.7,2.56,20.4,4.7,101.0,2.0,3685.0,74.0,0.33,1.92,1,-1.08,0.15,0.01,NaN
4,J005044.95-021217.6,2.25,21.5,4.7,147.0,9.0,4343.0,487.0,0.19,0.61,0,-1.73,0.14,0.02,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,J223754.52+065026.6,2.61,22.0,5.8,141.0,5.0,1391.0,44.0,0.36,1.03,0,-1.20,0.24,0.02,0.0
93,J225438.30+232714.5,3.09,22.0,5.5,415.0,20.0,4412.0,146.0,0.36,0.66,0,-0.38,NaN,NaN,NaN
94,J232326.17-010033.1,2.36,22.4,7.2,256.0,5.0,3989.0,62.0,0.37,2.16,0,0.91,0.40,0.01,0.0
95,J232828.47+044346.8,2.56,21.5,5.5,359.0,5.0,1584.0,20.0,0.35,0.37,0,-0.28,0.13,0.02,0.0


#### extract positions from identifier

In [ ]:
# def extract_ra_in_deg(identifier):
#     substring = identifier[1:10]
#     formatted_string = substring[0:2]+"h"+substring[2:4]+"m"+substring[4:]+"s"

#     ra = Angle(formatted_string, unit=u.hourangle)

#     return ra.deg

# def extract_dec_in_deg(identifier):
#     substring = identifier[10:]
#     formatted_string = substring[0:3]+"d"+substring[3:5]+"m"+substring[5:]+"s"

#     dec = Angle(formatted_string, unit=u.deg)

#     return dec.deg


# hamann_erq["RA"] = hamann_erq["SDSS_id"].apply(lambda x: extract_ra_in_deg(x))
# hamann_erq["DEC"] = hamann_erq["SDSS_id"].apply(lambda x: extract_dec_in_deg(x))


After removing the unusual minus sign characters from the CSV, using the `SkyCoord.from_name()` worked, which is the preferred option since the values were not entirely the same as when converting them myself.

In [ ]:
# hamann_erq["database_coords"] = hamann_erq["SDSS_id"].apply(lambda x: SkyCoord.from_name(f"SDSS {x}"))

# hamann_erq["RA"] = hamann_erq["database_coords"].apply(lambda x: x.ra.deg)
# hamann_erq["DEC"] = hamann_erq["database_coords"].apply(lambda x: x.dec.deg)
# hamann_erq = hamann_erq.drop(columns=["database_coords"])

# hamann_erq = hamann_erq[['SDSS_id', 'RA', 'DEC', 'ze', 'i', 'i-W3', 'REW', 'REW_unc', 'FWHM', 'FWHM_unc', 'kt80', 'NV/CIV', 'BAL', 'UV_slope', 'E(B-V)', 'E(B-V)_unc', 'FIRST']]

# hamann_erq

,SDSS_id,RA,DEC,ze,i,i-W3,REW,REW_unc,FWHM,FWHM_unc,kt80,NV/CIV,BAL,UV_slope,E(B-V),E(B-V)_unc,FIRST
0,J000610.67+121501.2,1.544491,12.250354,2.31,22.1,8.0,107.0,6.0,4540.0,200.0,0.37,1.88,0,-0.20,0.66,0.01,0.0
1,J000746.19+122223.9,1.942491,12.373315,2.43,21.3,4.9,220.0,6.0,2433.0,71.0,0.28,0.39,0,-0.17,0.19,0.01,0.0
2,J002400.25+245031.9,6.001042,24.842208,2.81,21.6,6.0,144.0,10.0,3523.0,195.0,0.37,2.46,0,-1.14,0.20,0.02,NaN
3,J004713.21+264024.7,11.805059,26.673525,2.56,20.4,4.7,101.0,2.0,3685.0,74.0,0.33,1.92,1,-1.08,0.15,0.01,NaN
4,J005044.95-021217.6,12.687326,-2.204915,2.25,21.5,4.7,147.0,9.0,4343.0,487.0,0.19,0.61,0,-1.73,0.14,0.02,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,J223754.52+065026.6,339.477177,6.840747,2.61,22.0,5.8,141.0,5.0,1391.0,44.0,0.36,1.03,0,-1.20,0.24,0.02,0.0
93,J225438.30+232714.5,343.659621,23.454052,3.09,22.0,5.5,415.0,20.0,4412.0,146.0,0.36,0.66,0,-0.38,NaN,NaN,NaN
94,J232326.17-010033.1,350.859062,-1.009201,2.36,22.4,7.2,256.0,5.0,3989.0,62.0,0.37,2.16,0,0.91,0.40,0.01,0.0
95,J232828.47+044346.8,352.118656,4.729677,2.56,21.5,5.5,359.0,5.0,1584.0,20.0,0.35,0.37,0,-0.28,0.13,0.02,0.0


### Find Hamann ERQ in DESI

In [2]:
# hamann_erq.to_csv(hamann_erq_path)

hamann_erq = pd.read_csv(hamann_erq_path, index_col=0)

hamann_erq

,SDSS_id,RA,DEC,ze,i,i-W3,REW,REW_unc,FWHM,FWHM_unc,kt80,NV/CIV,BAL,UV_slope,E(B-V),E(B-V)_unc,FIRST
0,J000610.67+121501.2,1.544491,12.250354,2.31,22.1,8.0,107.0,6.0,4540.0,200.0,0.37,1.88,0,-0.20,0.66,0.01,0.0
1,J000746.19+122223.9,1.942491,12.373315,2.43,21.3,4.9,220.0,6.0,2433.0,71.0,0.28,0.39,0,-0.17,0.19,0.01,0.0
2,J002400.25+245031.9,6.001042,24.842208,2.81,21.6,6.0,144.0,10.0,3523.0,195.0,0.37,2.46,0,-1.14,0.20,0.02,NaN
3,J004713.21+264024.7,11.805059,26.673525,2.56,20.4,4.7,101.0,2.0,3685.0,74.0,0.33,1.92,1,-1.08,0.15,0.01,NaN
4,J005044.95-021217.6,12.687326,-2.204915,2.25,21.5,4.7,147.0,9.0,4343.0,487.0,0.19,0.61,0,-1.73,0.14,0.02,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,J223754.52+065026.6,339.477177,6.840747,2.61,22.0,5.8,141.0,5.0,1391.0,44.0,0.36,1.03,0,-1.20,0.24,0.02,0.0
93,J225438.30+232714.5,343.659621,23.454052,3.09,22.0,5.5,415.0,20.0,4412.0,146.0,0.36,0.66,0,-0.38,NaN,NaN,NaN
94,J232326.17-010033.1,350.859062,-1.009201,2.36,22.4,7.2,256.0,5.0,3989.0,62.0,0.37,2.16,0,0.91,0.40,0.01,0.0
95,J232828.47+044346.8,352.118656,4.729677,2.56,21.5,5.5,359.0,5.0,1584.0,20.0,0.35,0.37,0,-0.28,0.13,0.02,0.0


In [3]:
def log_result_to_json(sdss_id,best_match_id,results_df, filepath=search_log_path):
    
    entry = {
        "timestamp":     datetime.utcnow().isoformat(),
        "sdss_id":       sdss_id,
        "best_match_id": best_match_id,
        "results_df":    results_df.to_dict(orient="records")
    }

    with open(filepath, "r") as f:
        log = json.load(f)

    log.append(entry)

    with open(filepath, "w") as f:
        json.dump(log, f, indent=2, default=str)
    
    print(f"Logged entry for {sdss_id}")

In [4]:
def query_desi_by_coords(df, z_deviation=0.2, coord_deviaion=0.05):
    '''
    Iterates through dataframe, for each element searches in DESI for the object with smallest difference in RA, DEC, and z (in that order of importance).
    '''

    for elem_idx in range(len(df)):
        # extract ra, dec, z and id for current element
        elem_ra = df["RA"][elem_idx]
        elem_dec = df["DEC"][elem_idx]
        elem_z = df["ze"][elem_idx]
        elem_sdss_id = df["SDSS_id"][elem_idx]

        # query for objects within set devations of current element 
        query = f"""
                SELECT targetid, z, target_ra, target_dec
                FROM desi_dr1.agnqso
                WHERE   (z < {elem_z}+{z_deviation}) AND (z > {elem_z}-{z_deviation})
                        AND (target_ra < {elem_ra}+{coord_deviaion}) AND (target_ra > {elem_ra}-{coord_deviaion})
                        AND (target_dec < {elem_dec}+{coord_deviaion}) AND (target_dec > {elem_dec}-{coord_deviaion})
                """

        results = qc.query(sql=query, fmt="pandas")

        # if there are results, calculate differences, then sort to get best match
        if len(results) > 0:
            results["z_diff"] = abs(results["z"] - elem_z)
            results["ra_diff"] = abs(results["target_ra"] - elem_ra)
            results["dec_diff"] = abs(results["target_dec"] - elem_dec)
            results = results.sort_values(["ra_diff","dec_diff","z_diff"], ascending=[True,True,True])

            best_match = results["targetid"][0]
        # if not found, say so
        else:
            best_match = "Not found"            

        # log to json file
        log_result_to_json(elem_sdss_id,best_match,results)


# query_desi_by_coords(hamann_erq)